In [53]:
"""
Notebook: data_preprocessing.ipynb
Objetivo: Limpieza, transformación y preprocesamiento de los datos
del dataset BRFSS2015 del CDC para el posterior entrenamiento de
modelos de ML en detección de diabetes.

Autor: Jesús Rodríguez
Fecha: 06/11/2025
"""

'\nNotebook: data_preprocessing.ipynb\nObjetivo: Limpieza, transformación y preprocesamiento de los datos\ndel dataset BRFSS2015 del CDC para el posterior entrenamiento de \nmodelos de ML en detección de diabetes.\n\nAutor: Jesús Rodríguez\nFecha: 06/11/2025\n'

## 1. Configuración inicial

In [54]:
# Librerías básicas
from pathlib import Path
import pandas as pd
import numpy as np


# Interpretabilidad
from IPython.display import display

# Google Drive
from google.colab import drive

In [55]:
# Montaje de Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
# Definición de rutas
BASE_PATH = Path('/content/drive/MyDrive/TFM/data')
RAW_CSV_PATH = BASE_PATH / 'raw_dataset.csv'

# Carga del dataset
df_raw = pd.read_csv(
    RAW_CSV_PATH,
    encoding='latin-1'
)

In [57]:
# Información general del dataset
print(f"Shape del dataset: {df_raw.shape}")

df_raw.info()
df_raw.describe().transpose()

Shape del dataset: (441456, 330)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441456 entries, 0 to 441455
Columns: 330 entries, _STATE to _AIDTST3
dtypes: float64(324), int64(4), object(2)
memory usage: 1.1+ GB


,count,mean,std,min,25%,50%,75%,max
_STATE,441456.0,2.996871e+01,1.603471e+01,1.0,19.0,29.0,44.0,72.0
FMONTH,441456.0,6.359676e+00,3.487131e+00,1.0,3.0,6.0,9.0,12.0
IDATE,441456.0,6.563745e+06,3.488675e+06,1012016.0,3232015.0,6242015.0,10022015.0,12312015.0
IMONTH,441456.0,6.416803e+00,3.492082e+00,1.0,3.0,6.0,10.0,12.0
IDAY,441456.0,1.449273e+01,8.335468e+00,1.0,8.0,14.0,21.0,31.0
...,...,...,...,...,...,...,...,...
_RFSEAT2,441456.0,1.824624e+00,2.360812e+00,1.0,1.0,1.0,1.0,9.0
_RFSEAT3,441456.0,1.887028e+00,2.351387e+00,1.0,1.0,1.0,1.0,9.0
_FLSHOT6,157954.0,2.290705e+00,2.518086e+00,1.0,1.0,1.0,2.0,9.0
_PNEUMO2,157954.0,2.412259e+00,2.778032e+00,1.0,1.0,1.0,2.0,9.0


## 2. Análisis exploratorio inicial del target

In [58]:
# Distribución absoluta y relativa del target
target_counts = df_raw['DIABETE3'].value_counts()
target_pct = df_raw['DIABETE3'].value_counts(normalize=True).mul(100)

target_distribution = (  # pylint: disable=C0103
    pd.DataFrame({
        'Conteo': target_counts,
        'Porcentaje (%)': target_pct
    })
    .sort_index()
)
# Se deshabilita C0103 porque target_distribution no sigue UPPER_CASE,
# pero es más legible en análisis de datos.

print("Distribución de clases:")
print(target_distribution)

Distribución de clases:
          Conteo  Porcentaje (%)
DIABETE3                        
1.0        57256       12.970015
2.0         3608        0.817308
3.0       372104       84.291504
4.0         7690        1.741991
7.0          598        0.135463
9.0          193        0.043720


## 3. Cribado de columnas

In [59]:
# Se conservan únicamente las clases relevantes del target
TARGET_CLASSES = [1, 3]

df_raw = (
    df_raw[df_raw['DIABETE3'].isin(TARGET_CLASSES)]
    .reset_index(drop=True)
    .copy()
)

### 3.1. Primer cribado de columnas (330 variables)

In [60]:
# Eliminación de registros duplicados
df_raw = df_raw.drop_duplicates().copy()

In [61]:
# Definición de columnas irrelevantes para la detección de diabetes
COLS_TO_DROP_STAGE_1 = [
    '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO',
    '_PSU', 'QSTVER', 'QSTLANG', 'MSCODE', '_STSTR', '_STRWT', '_RAWRAKE',
    '_WT2RAKE', '_CLLCPWT', '_DUALUSE', '_DUALCOR', '_LLCPWT', 'CTELENUM',
    'CELLFON3', 'CTELNUM1', 'CELLFON2', 'CADULT', 'CCLGHOUS', 'CSTATE',
    'LANDLINE', 'NUMPHON2', 'CPDEMO1', 'INTERNET', 'PVTRESD1', 'COLGHOUS',
    'STATERES', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'PVTRESD2',
    'HHADULT', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'RENTHOM1',
    'NUMHHOL2', 'CHILDREN', 'INCOME2', 'CDHELP', 'CDSOCIAL', '_CHLDCNT',
    '_INCOMG', 'DIABAGE2', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'CHCKIDNY',
    'PDIABTST', 'PREDIAB1', 'INSULIN', 'FEETCHK2', 'DOCTDIAB', 'CHKHEMO3',
    'FEETCHK', 'EYEEXAM', 'DIABEYE', 'DIABEDU', 'MARITAL', 'EDUCA','VETERAN3',
    'EMPLOY1', 'SEATBELT', 'CDDISCUS', 'SCNTMNY1', 'SCNTMEL1', 'SCNTPAID',
    'SCNTWRK1', 'SCNTLPAD', 'SCNTLWK1', 'SXORIENT', 'TRNSGNDR', 'RCSGENDR',
    'RCSRLTN2', 'EMTSUPRT', 'LSATISFY', 'ADPLEASR', 'ADDOWN', 'MISTMNT',
    '_EDUCAG', '_RFSEAT2', '_RFSEAT3', '_LMTWRK1', '_LMTSCL1', 'CAREGIV1',
    'CRGVREL1', 'CRGVLNG1', 'CRGVHRS1', 'CRGVPRB1', 'CRGVPERS', 'CRGVHOUS',
    'CRGVMST2', 'CRGVEXPT', 'CDHOUSE', 'CDASSIST', 'WEIGHT2', 'HEIGHT3',
    'HTIN4', 'HTM4', 'WTKG3', '_BMI5', '_RFBMI5', 'STOPSMK2', 'LASTSMK2',
    'USENOW3', '_SMOKER3', '_RFSMOK3', 'ALCDAY5', 'DRNK3GE5', 'MAXDRNKS',
    'DRNKANY5', 'DROCDY3_', '_RFBING5', '_DRNKWEK', '_RFDRHV5', 'FVGREEN',
    'ARTHEXER', 'FVORANG', 'VEGETAB1', 'GRENDAY_', 'ORNGDAY_', 'VEGEDA1_',
    '_MISVEGN', '_VEGLT1', '_VEG23', '_VEGETEX', 'FTJUDA1_', 'FRUTDA1_',
    'FRUITJU1', 'FRUIT1', '_MISFRTN', '_FRTLT1', '_FRT16', '_FRUITEX',
    'FVBEANS', 'EXERANY2', 'EXRACT11', 'EXRACT21', 'EXEROFT2', 'EXERHMM2',
    'ADMOVE', 'EXACTOT1', 'EXACTOT2', '_TOTINDA', 'METVL11_', 'METVL21_',
    'MAXVO2_', 'FC60_', 'ACTIN11_', 'ACTIN21_', 'PADUR1_', 'PADUR2_',
    'PAFREQ1_', 'PAFREQ2_', '_MINAC11', '_MINAC21', 'STRFREQ_', 'PAMIN11_',
    'PAMIN21_', 'PA1MIN_', 'PAVIG11_', 'PAVIG21_', 'PA1VIGM_', '_PA150R2',
    '_PA300R2', '_PA30021', '_PASTRNG', '_PAREC1', '_PASTAE1', '_LMTACT1',
    'LMTJOIN3', 'ARTHSOCL', 'JOINPAIN', 'PAINACT2', 'TOLDHI2', 'QLMENTL2',
    'QLSTRES2', 'QLHLTH2', '_RFHLTH', 'VIDFCLT2', 'VIREDIF3', 'VIPRFVS2',
    'VINOCRE2', 'VIEYEXM2', 'VIINSUR2', 'VICTRCT4', 'VIGLUMA2', 'VIMACDG2',
    'ASTHMAGE', 'ASATTACK', 'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM',
    'ASYMPTOM', 'ASNOSLEP', 'ASTHMED3', 'ASINHALR', 'CASTHDX2', 'CASTHNO2',
    '_LTASTH1', '_CASTHM1', '_ASTHMS1', 'HAREHAB1', 'STREHAB1', 'CVDASPRN',
    'ASPUNSAF', '_MICHD', 'RLIVPAIN', 'RDUCHART', 'RDUCSTRK', 'ARTTODAY',
    'ARTHWGT', 'ARTHEDU', '_DRDXAR1', '_RFCHOL',  '_CHISPNC', '_CRACE1',
    '_CPRACE', '_PRACE1', '_MRACE1', '_HISPANC', '_RACEG21', '_RACEGR3',
    '_RACE_G1', '_AGE65YR', '_AGE80', '_AGE_G', '_RFHYPE5', 'CHOLCHK',
    'HIVTST6', 'HIVTSTD3', 'WHRTST10', 'HADMAM', 'HOWLONG', 'HADPAP2',
    'LASTPAP2', 'HPVTEST', 'HPLSTTST', 'HADHYST2', 'PROFEXAM', 'LENGEXAM',
    'BLDSTOOL', 'LSTBLDS3', 'HADSIGM3', 'HADSGCO1', 'LASTSIG3', 'PSATEST1',
    'PSATIME', 'PCPSARS1', 'PCPSADE1', 'PCDMDECN', '_AIDTST3', '_CHOLCHK',
    'FLUSHOT6', 'FLSHTMY2', 'IMFVPLAC', 'PNEUVAC3', 'TETANUS', 'HPVADVC2',
    'HPVADSHT', 'SHINGLE2', '_FLSHOT6', '_PNEUMO2', 'DRADVISE', 'PREGNANT',
    'PCPSAAD2', 'PCPSADI1', 'PCPSARE1', '_HCVU651', 'STRENGTH', 'PAMISS1_',
    'SMOKDAY2', 'EXERHMM1', 'WTCHSALT', 'LONGWTCH', 'BEANDAY_', '_FRTRESP',
    '_VEGRESP', '_PAINDX1', 'CIMEMLOS', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR',
    'CHCOCNCR', 'CHCCOPD1', 'BLDSUGAR'
]

In [62]:
# Eliminación de columnas irrelevantes
df_cleaned_1 = (
    df_raw
    .drop(columns=COLS_TO_DROP_STAGE_1, errors='ignore')
    .copy()
)

In [63]:
# Comprobación de consistencia del número total de columnas
total_columns = df_cleaned_1.shape[1] + len(COLS_TO_DROP_STAGE_1)
print(f'Total de columnas consideradas en el cribado: {total_columns}')

Total de columnas consideradas en el cribado: 330


### 3.2. Segundo cribado de columnas (34 variables)

In [64]:
# Cálculo del número y porcentaje de valores nulos por columna
nulls = (
    df_cleaned_1
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

nulls_pct = (
    df_cleaned_1
    .isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

nulls_summary = pd.DataFrame({
    'Nulos': nulls,
    'Porcentaje (%)': nulls_pct
})

display(nulls_summary)

,Nulos,Porcentaje (%)
ADANXEV,409608,95.399665
ADTHINK,409571,95.391047
ADFAIL,409557,95.387786
ADEAT1,409550,95.386156
ADENERGY,409539,95.383594
ADSLEEP,409534,95.382430
ARTHDIS2,297604,69.313397
BPMEDS,257357,59.939678
AVEDRNK2,223367,52.023244
POORHLTH,209652,48.828955


In [65]:
# Columnas con elevado número de NaN o menor relevancia clínica
COLS_TO_DROP_STAGE_2 = [
    'ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL',
    'ADTHINK', 'ADANXEV', 'ARTHDIS2',
    'POORHLTH', 'AVEDRNK2'
    ]

In [66]:
# Eliminación de columnas seleccionadas
df_cleaned_2 = (
    df_cleaned_1
    .drop(columns=COLS_TO_DROP_STAGE_2, errors='ignore')
    .copy()
)

In [67]:
# Comprobación de consistencia del número total de columnas
total_columns = df_cleaned_2.shape[1] + len(COLS_TO_DROP_STAGE_2)
print(f'Total de columnas consideradas en el cribado: {total_columns}')

Total de columnas consideradas en el cribado: 34


### 3.3. Tercer cribado de columnas (25 variables)

In [68]:
class MissingValueAnalyzer:
    """
    Analiza porcentajes de valores especiales en un DataFrame, que representan
    respuestas inválidas o no informadas.
    """

    def __init__(self, df: pd.DataFrame):
        """Inicializa con un DataFrame."""
        self.df = df

    def calculate_percentage(
        self, columns: list[str], special_values: list
    ) -> pd.DataFrame:
        """
        Calcula el porcentaje de valores especiales por columna.

        Args:
            columns: Columnas a analizar.
            special_values: Valores considerados especiales.

        Returns:
            DataFrame con columnas 'Columna' y 'Porcentaje (%)'.
        """
        results = [
            {
                "Columna": col,
                "Porcentaje (%)": (self.df[col].isin(special_values).sum() /
                                   max(self.df[col].notna().sum(), 1) * 100)
            }
            for col in columns
        ]
        return pd.DataFrame(results)

    def calculate_all_groups(self, column_groups: dict) -> dict:
        """
        Calcula porcentajes para varios grupos de columnas.

        Args:
            column_groups: dict {nombre_grupo: (columnas, valores_especiales)}

        Returns:
            dict con DataFrames como valores.
        """
        return {
            name: self.calculate_percentage(cols, values)
            for name, (cols, values) in column_groups.items()
        }

In [69]:
# Creación del analizador de valores faltantes
missing_value_analyzer = MissingValueAnalyzer(df_cleaned_2)

In [70]:
# Definición de grupos de columnas y valores especiales
COLUMN_GROUPS = {
    '7_9': (
        ['GENHLTH', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3',
         'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK',
         'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_PACAT1'],
        [7, 9]
    ),
    '77_88_99': (
        ['PHYSHLTH', 'MENTHLTH'],
        [77, 88, 99]
    ),
    '777_999': (
        ['EXEROFT1'],
        [777, 999]
    ),
    '9': (
        ['_RACE'],
        [9]
    ),
    '14': (
        ['_AGEG5YR'],
        [14]
    )
}

In [71]:
# Cálculo de resultados por grupo
missing_value_results = missing_value_analyzer.calculate_all_groups(COLUMN_GROUPS)

In [72]:
# Visualización de resultados
for group_name, df_result in missing_value_results.items():
    print(f"\n=== Resultados para grupo {group_name} ===")
    print(df_result)


=== Resultados para grupo 7_9 ===
     Columna  Porcentaje (%)
0    GENHLTH        0.277158
1    BPHIGH4        0.285542
2     BPMEDS        0.171509
3   BLOODCHO        2.138066
4   HAVARTH3        0.587853
5   QLACTLM2        0.715295
6   USEEQUIP        0.248342
7      BLIND        0.377324
8     DECIDE        0.659482
9   DIFFWALK        0.514461
10  DIFFDRES        0.264543
11  DIFFALON        0.431115
12  SMOKE100        0.758683
13  ADDEPEV2        0.457192
14   _PACAT1       12.667691

=== Resultados para grupo 77_88_99 ===
    Columna  Porcentaje (%)
0  PHYSHLTH       64.525257
1  MENTHLTH       70.061720

=== Resultados para grupo 777_999 ===
    Columna  Porcentaje (%)
0  EXEROFT1        0.960703

=== Resultados para grupo 9 ===
  Columna  Porcentaje (%)
0   _RACE          1.6669

=== Resultados para grupo 14 ===
    Columna  Porcentaje (%)
0  _AGEG5YR        1.194336


In [73]:
# Eliminación de columnas con valores especiales significativos
COLS_TO_DROP_STAGE_3 = ['PHYSHLTH', 'MENTHLTH']

df_cleaned_3 = (
    df_cleaned_2
    .drop(columns=COLS_TO_DROP_STAGE_3, errors='ignore')
    .copy()
)

In [74]:
# Comprobaciones finales de consistencia
total_columns = df_cleaned_3.shape[1] + len(COLS_TO_DROP_STAGE_3)

print(f'Total de columnas consideradas en el cribado: {total_columns}')
print(f'Número final de columnas: {df_cleaned_3.shape[1]}')

Total de columnas consideradas en el cribado: 25
Número final de columnas: 23


## 4. Transformación e imputación de variables

### 4.1. Variables que podrían estar relacionadas

#### BPMEDS y su relación con BPHIGH

In [75]:
# Indicador auxiliar de valores nulos en BPMEDS
df_cleaned_3['BPMEDS_is_null'] = df_cleaned_3['BPMEDS'].isna()

In [76]:
# Distribución de nulos de BPMEDS según hipertensión diagnosticada
bphigh_distribution = pd.crosstab(
    df_cleaned_3['BPMEDS_is_null'],
    df_cleaned_3['BPHIGH4'],
    margins=True,
    normalize='columns'
)

print("Distribución de BPMEDS nulos según BPHIGH4:")
print(bphigh_distribution)

Distribución de BPMEDS nulos según BPHIGH4:
BPHIGH4         1.0  2.0  3.0  4.0  7.0  9.0       All
BPMEDS_is_null                                        
False           1.0  0.0  0.0  0.0  0.0  0.0  0.400604
True            0.0  1.0  1.0  1.0  1.0  1.0  0.599396


In [77]:
# Imputación clínica:
# Si no tiene hipertensión, no toma medicación → categoría 2 (NO)
df_cleaned_3['BPMEDS'] = df_cleaned_3['BPMEDS'].fillna(2)

In [78]:
# Verificación de imputación
print(f"Número de NaN restantes en BPMEDS: {df_cleaned_3['BPMEDS'].isna().sum()}")

Número de NaN restantes en BPMEDS: 0


In [79]:
# Eliminación de la variable auxiliar
df_cleaned_3.drop(columns='BPMEDS_is_null', inplace=True)

#### EXEROFT1 y limitaciones de movilidad

In [80]:
# Indicador auxiliar de valores nulos en EXEROFT1
df_cleaned_3['EXEROFT1_is_null'] = df_cleaned_3['EXEROFT1'].isna()

In [81]:
# Análisis de relación entre nulos en EXEROFT1 y limitaciones funcionales
for variable in ['DIFFALON', 'DIFFDRES', 'DIFFWALK']:
    crosstab = pd.crosstab(
        df_cleaned_3['EXEROFT1_is_null'],
        df_cleaned_3[variable],
        margins=True,
        normalize='columns'
    )

    print(f"\nDistribución de nulos de EXEROFT1 según {variable}:")
    print(crosstab)


Distribución de nulos de EXEROFT1 según DIFFALON:
DIFFALON               1.0       2.0       7.0       9.0    All
EXEROFT1_is_null                                               
False             0.431847  0.711394  0.475485  0.180828  0.688
True              0.568153  0.288606  0.524515  0.819172  0.312

Distribución de nulos de EXEROFT1 según DIFFDRES:
DIFFDRES               1.0       2.0       7.0       9.0       All
EXEROFT1_is_null                                                  
False             0.398142  0.701415  0.436314  0.095368  0.687038
True              0.601858  0.298585  0.563686  0.904632  0.312962

Distribución de nulos de EXEROFT1 según DIFFWALK:
DIFFWALK               1.0       2.0       7.0       9.0       All
EXEROFT1_is_null                                                  
False             0.465478  0.734292  0.565015  0.177778  0.686392
True              0.534522  0.265708  0.434985  0.822222  0.313608


In [82]:
# Conclusión: los NaN no tienen un significado clínico claro
# Se elimina la variable auxiliar
df_cleaned_3.drop(columns='EXEROFT1_is_null', inplace=True)

### 4.2. Tratamiento de variables especiales

#### EXEROFT1 – decodificación de frecuencia de ejercicio

In [83]:
def decode_exeroft1(value):
    """
    Decodifica la variable EXEROFT1:
    - 101–199 → veces por semana
    - 201–299 → veces por mes (convertido a semana)
    - 777, 999 → No sabe / No contesta (se conservan)
    """
    if pd.isna(value):
        return np.nan

    if 101 <= value <= 199:
        return value - 100

    if 201 <= value <= 299:
        return (value - 200) / (30.0 / 7.0)

    if value in (777, 999):
        return value

    return np.nan

In [84]:
# Aplicación de la transformación
df_cleaned_3['EXEROFT1'] = df_cleaned_3['EXEROFT1'].apply(decode_exeroft1)

In [85]:
# Inspección de valores tras la transformación
df_cleaned_3['EXEROFT1'].round(2).unique()

array([      nan, 2.800e+00, 1.000e+00, 2.000e+00, 6.000e+00, 3.000e+00,
       4.670e+00, 7.000e+00, 7.770e+02, 1.870e+00, 2.330e+00, 5.000e+00,
       4.000e+00, 3.500e+00, 1.170e+00, 7.000e-01, 2.300e-01, 4.700e-01,
       1.400e+01, 1.400e+00, 9.300e-01, 3.730e+00, 1.630e+00, 6.530e+00,
       5.830e+00, 4.200e+00, 6.770e+00, 1.000e+01, 9.990e+02, 6.300e+00,
       4.900e+00, 5.600e+00, 2.100e+01, 3.270e+00, 3.030e+00, 6.070e+00,
       2.100e+00, 5.370e+00, 8.000e+00, 5.130e+00, 3.970e+00, 2.310e+01,
       1.500e+01, 2.000e+01, 2.400e+01, 5.600e+01, 2.570e+00, 9.330e+00,
       7.230e+00, 1.050e+01, 8.170e+00, 1.200e+01, 2.500e+01, 3.000e+01,
       1.167e+01, 1.800e+01, 3.500e+01, 9.000e+00, 1.100e+01, 5.000e+01,
       9.800e+00, 2.800e+01, 1.700e+01, 4.430e+00, 4.200e+01, 4.000e+01,
       7.470e+00, 1.750e+01, 4.900e+01, 1.600e+01, 7.000e+01, 1.867e+01,
       8.400e+00, 3.100e+01, 1.300e+01, 4.300e+01, 7.500e+01, 8.630e+00,
       3.400e+01, 1.633e+01, 9.900e+01, 1.517e+01, 

#### _FRUTSUM y _VEGESUM – normalización de consumo diario

In [86]:
# Conversión de raciones * 100 a raciones reales por día
df_cleaned_3['_FRUTSUM'] = df_cleaned_3['_FRUTSUM'] / 100
df_cleaned_3['_VEGESUM'] = df_cleaned_3['_VEGESUM'] / 100

In [87]:
# Verificación de valores transformados
print(df_cleaned_3['_FRUTSUM'].round(2).unique())
print(df_cleaned_3['_VEGESUM'].round(2).unique())

[5.000e-01 2.400e-01       nan 1.000e+00 2.000e+00 1.290e+00 3.000e+00
 1.140e+00 4.300e-01 8.600e-01 3.400e-01 7.000e-02 1.670e+00 1.430e+00
 2.900e-01 0.000e+00 1.000e-01 1.700e-01 3.600e-01 1.570e+00 4.000e+00
 1.330e+00 6.700e-01 1.070e+00 2.290e+00 3.000e-02 5.300e-01 6.000e-01
 8.300e-01 1.500e+00 2.000e-01 2.600e-01 5.700e-01 4.290e+00 2.140e+00
 3.000e-01 1.830e+00 1.230e+00 5.000e+00 3.570e+00 1.600e-01 7.600e-01
 3.300e-01 6.200e-01 5.130e+00 3.700e-01 1.100e+00 7.200e-01 4.140e+00
 4.400e-01 1.380e+00 1.400e-01 1.300e-01 4.200e-01 7.400e-01 2.430e+00
 8.400e-01 7.800e-01 5.800e-01 1.170e+00 2.000e-02 7.000e+00 7.100e-01
 2.100e-01 1.030e+00 4.000e-01 5.600e-01 2.800e-01 1.130e+00 1.860e+00
 6.000e+00 5.290e+00 1.710e+00 2.070e+00 6.000e-02 8.700e-01 4.700e-01
 9.000e-01 1.280e+00 6.600e-01 3.200e-01 2.300e-01 4.600e-01 2.700e-01
 1.200e+00 3.330e+00 3.900e-01 1.160e+00 8.000e-01 4.860e+00 9.600e-01
 7.300e-01 2.670e+00 3.430e+00 3.100e-01 1.730e+00 7.000e-01 2.570e+00
 2.170

### 4.3. Imputación

#### Eliminación inicial de NaN estructurales

In [88]:
# Eliminación de registros con NaN reales (estructurales)
df_cleaned_3 = df_cleaned_3.dropna()

In [89]:
# Verificación del estado del dataset
df_cleaned_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 257709 entries, 1 to 429359
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   257709 non-null  float64
 1   BPHIGH4   257709 non-null  float64
 2   BPMEDS    257709 non-null  float64
 3   BLOODCHO  257709 non-null  float64
 4   HAVARTH3  257709 non-null  float64
 5   ADDEPEV2  257709 non-null  float64
 6   DIABETE3  257709 non-null  float64
 7   SEX       257709 non-null  float64
 8   QLACTLM2  257709 non-null  float64
 9   USEEQUIP  257709 non-null  float64
 10  BLIND     257709 non-null  float64
 11  DECIDE    257709 non-null  float64
 12  DIFFWALK  257709 non-null  float64
 13  DIFFDRES  257709 non-null  float64
 14  DIFFALON  257709 non-null  float64
 15  SMOKE100  257709 non-null  float64
 16  EXEROFT1  257709 non-null  float64
 17  _RACE     257709 non-null  float64
 18  _AGEG5YR  257709 non-null  float64
 19  _BMI5CAT  257709 non-null  float64
 20  _FRUTSUM 

#### Identificación de valores especiales (“No sabe / Se negó”)

In [90]:
# Valores especiales que representan respuestas inválidas
# (_BMI5CAT, _FRUTSUM, _VEGESUM y SEX no presentan estas categorías)
# DIABETE3 ya fue filtrada previamente
INVALID_VALUES = {
    'GENHLTH': [7, 9],
    'BPHIGH4': [7, 9],
    'BPMEDS': [7, 9],
    'BLOODCHO': [7, 9],
    'HAVARTH3': [7, 9],
    'QLACTLM2': [7, 9],
    'USEEQUIP': [7, 9],
    'BLIND': [7, 9],
    'DECIDE': [7, 9],
    'DIFFWALK': [7, 9],
    'DIFFDRES': [7, 9],
    'DIFFALON': [7, 9],
    'SMOKE100': [7, 9],
    'ADDEPEV2': [7, 9],
    '_PACAT1': [9],
    '_RACE': [9],
    '_AGEG5YR': [14],
    'EXEROFT1': [777, 999]
}

In [91]:
# Sustitución de valores especiales por NaN
for column, invalid_values in INVALID_VALUES.items():
    df_cleaned_3[column] = df_cleaned_3[column].replace(invalid_values, np.nan)

#### Separación por tipo de variable

In [92]:
# Definición de tipos de variables
CATEGORICAL_NOMINAL = [
    'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2',
    'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES',
    'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_RACE'
]

CATEGORICAL_ORDINAL = [
    'GENHLTH', '_PACAT1', '_AGEG5YR'
]

NUMERICAL = [
    'EXEROFT1'
]

#### Imputación

In [93]:
# Imputación de variables categóricas (nominales y ordinales)
for column in CATEGORICAL_NOMINAL + CATEGORICAL_ORDINAL:
    df_cleaned_3[column] = df_cleaned_3[column].fillna(-1)

In [94]:
# Imputación de variables numéricas mediante la mediana
for column in NUMERICAL:
    df_cleaned_3[column] = df_cleaned_3[column].fillna(
        df_cleaned_3[column].median()
    )

In [95]:
# Reindexado final del dataset
df_cleaned_3 = df_cleaned_3.reset_index(drop=True)

## 5. Exportación

In [96]:
# Verificación final del dataset antes de exportar
df_cleaned_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 257709 entries, 0 to 257708
Data columns (total 23 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   GENHLTH   257709 non-null  float64
 1   BPHIGH4   257709 non-null  float64
 2   BPMEDS    257709 non-null  float64
 3   BLOODCHO  257709 non-null  float64
 4   HAVARTH3  257709 non-null  float64
 5   ADDEPEV2  257709 non-null  float64
 6   DIABETE3  257709 non-null  float64
 7   SEX       257709 non-null  float64
 8   QLACTLM2  257709 non-null  float64
 9   USEEQUIP  257709 non-null  float64
 10  BLIND     257709 non-null  float64
 11  DECIDE    257709 non-null  float64
 12  DIFFWALK  257709 non-null  float64
 13  DIFFDRES  257709 non-null  float64
 14  DIFFALON  257709 non-null  float64
 15  SMOKE100  257709 non-null  float64
 16  EXEROFT1  257709 non-null  float64
 17  _RACE     257709 non-null  float64
 18  _AGEG5YR  257709 non-null  float64
 19  _BMI5CAT  257709 non-null  float64
 20  _FRU

In [97]:
# Exportación del dataset limpio
OUTPUT_PATH = "/content/drive/MyDrive/TFM/data/cleaned_dataset.csv"
df_cleaned_3.to_csv(OUTPUT_PATH, index=False)